<a href="https://colab.research.google.com/github/jonibekjumaev/computer_vision/blob/main/menu_detector_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
print('Menu Detector')

Menu Detector


In [3]:
#---------------------
# Import Libraries   -
#---------------------

from google.colab import drive
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.models import mobilenet_v2

from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset, DataLoader
import os
import numpy as np

In [4]:
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# Define Dataset Path

DATASET_PATH = '/content/drive/MyDrive/food101_dataset'
print('Dataset_path:', DATASET_PATH)

CUSTOM_CLASS_MAPPING = {
    'hamburger': 'hamburger',
    'hot_dog': 'hot_dog',
    'chocolate_cake': 'dessert',  # label grouping | class consolidation
    'cheesecake': 'dessert',       # label grouping | class consolidation
    'kebab': 'kebab',
    'pilaf': 'pilaf'
}


CLASSES = ['hamburger', 'hot_dog', 'dessert', 'kebab', 'pilaf']
CLASS_TO_IDX = {cls: i for i, cls in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)

print(CLASS_TO_IDX)
print(NUM_CLASSES)


# preprocessing
transform = transforms.Compose([   # ketma-ket bajaradigan pipeline
    transforms.Resize((224,224)),
    transforms.ToTensor(),  # rasmni tensor’ga aylantirish    # H, W, C => C, H, W | RGB = Red, Green, Blue
    transforms.Normalize( mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225]) # pixel = (pixel - mean) / std
])

# 0 ~ 255
# 0.0 ~ 1.0
# 0.234
# H, W, C => C, H, W | RGB = Red, Green, Blue
# Normalize
# pixel = (pixel - mean) / std


Dataset_path: /content/drive/MyDrive/food101_dataset
{'hamburger': 0, 'hot_dog': 1, 'dessert': 2, 'kebab': 3, 'pilaf': 4}
5


In [16]:
# ----------------------
# Custom Dataset Class
# ----------------------

class FoodDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        # print('images_length', len(self.images))
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        # print('image_path', img_path)
        label = self.labels[idx]
        # print('label', label)
        try:
            image = Image.open(img_path)
            if image.mode == "P" or image.mode == "RGBA": # png | gif | RGBA rasmlar bulsa muammo bulmaydi
                image = image.convert("RGBA").convert("RGB")
            else:
                image = image.convert("RGB")
        except (UnidentifiedImageError, OSError):
            # print(f"Skipping broken image: {img_path}")
            return self.__getitem__((idx + 1) % len(self.images))
        if self.transform:
            image = self.transform(image)
        return image, label

In [7]:
# =========================
# Gather and Split Data
# =========================
all_images = []
for original_class, mapped_class in CUSTOM_CLASS_MAPPING.items():
    class_path = os.path.join(DATASET_PATH, original_class)  # /content/drive/MyDrive/food101_dataset/hamburger
    # print('class_path:', class_path)
    if not os.path.exists(class_path):
        print(f"Warning: {class_path} not found")
        continue
    for img in os.listdir(class_path):
        if img.endswith(('.jpg', '.jpeg', '.png')): # .txt .docs
            full_path = os.path.join(class_path, img)
            all_images.append((full_path, CLASS_TO_IDX[mapped_class]))

np.random.shuffle(all_images)
split = int(0.8 * len(all_images))
train_data = all_images[:split]
val_data = all_images[split:]

train_images, train_labels = zip(*train_data)
val_images, val_labels = zip(*val_data)

# print('all_images:', all_images)

dataset = FoodDataset(train_images, train_labels)
# print(len(dataset))
img, lbl = dataset[0]

class_path: /content/drive/MyDrive/food101_dataset/hamburger
class_path: /content/drive/MyDrive/food101_dataset/hot_dog
class_path: /content/drive/MyDrive/food101_dataset/chocolate_cake
class_path: /content/drive/MyDrive/food101_dataset/cheesecake
class_path: /content/drive/MyDrive/food101_dataset/kebab
class_path: /content/drive/MyDrive/food101_dataset/pilaf
images_length 3232
3232
image_path /content/drive/MyDrive/food101_dataset/cheesecake/1656988.jpg
label 2


In [8]:
train_dataset = FoodDataset(train_images, train_labels, transform=transform)
val_dataset = FoodDataset(val_images, val_labels, transform=transform)

In [9]:
train_loader = DataLoader(train_dataset,
                          batch_size=32, # birdan 32ta rasmni beradi train uchun
                          shuffle=True,  # har epoch’da rasmlar tartibi yangidan aralashtiriladi, shunda model tartibni “yodlab” olmaydi
                          num_workers=2) # ikkita worker thread ishlaydi | parallel loading for speed
                          # rasmlarni ikkita alohida process yuklaydi, GPU model’ni hisoblayotganda keyingi batch parallel ravishda tayyorlanadi
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

images_length 3232
images_length 3232


In [10]:
# batch_size=32: model’ga bir vaqtda 32 ta rasm beriladi. Natijada bitta batch’ning shape’i [32, 3, 224, 224] bo’ladi
# shuffle=True: har epoch’da rasmlar tartibi yangidan aralashtiriladi, shunda model tartibni “yodlab” olmaydi
# num_workers=2: rasmlarni ikkita alohida process yuklaydi. Izohingizda “thread” deb yozilgan, lekin PyTorch DataLoader worker’lari alohida process’lar sifatida ishlaydi. Natija bir xil: GPU model’ni hisoblayotganda keyingi batch parallel ravishda tayyorlanadi

In [11]:
# CNN - Convolutional Neural Network in Machine Learning

In [12]:
# pretrained model
model = mobilenet_v2(weights="IMAGENET1K_V1")  # pretrained model | lightweight | CNN | 1000 class
model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES) # fine-tuning faqat num_classesda berilgan class buyicha train buladi

Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 187MB/s]


In [13]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
model = model.to(device)

device: cuda


In [14]:
criterion = nn.CrossEntropyLoss()  # Loss Function | '70%' burger, '30%' pilaf
optimizer = optim.Adam(model.parameters(), lr=0.001)  # weight
torch.backends.cudnn.benchmark = True  # Benchmark Setting | Trick | 10%-20% mos gpu ni topib tezlashtiradi

In [15]:
#---------------------
# Training Loop  -
#---------------------

NUM_EPOCHS = 10
best_accuracy = 0.0

for epoch in range(NUM_EPOCHS):
    model.train()  # train mode
    running_loss = 0.0  # 70% | 30%Loss | 100%

    for images, labels in train_loader:  # Forward and Backward(Backpropagation)
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()  # zero the gradient
        outputs = model(images)  # Forward Pass | Dog | 5 Classes
        loss = criterion(outputs, labels)  # Calculate Loss
        loss.backward()
        optimizer.step()  # Adam optimizer
        running_loss += loss.item()  # Track Loss

    # Validation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_acc = 100 * correct / total  # Calculate Validation Accuracy
    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] Loss: {running_loss/len(train_loader):.4f}, Val Accuracy: {val_acc:.2f}%")

    if val_acc > best_accuracy:
        best_accuracy = val_acc
        torch.save(model.state_dict(), '/content/menu_detector.pth')
        print("Saved new best model!")

image_pathimage_path  /content/drive/MyDrive/food101_dataset/cheesecake/311588.jpg/content/drive/MyDrive/food101_dataset/cheesecake/2928889.jpg

labellabel  22

images_length 3232
images_length 3232
image_path /content/drive/MyDrive/food101_dataset/hot_dog/3835641.jpg
label 1
image_path /content/drive/MyDrive/food101_dataset/cheesecake/2103699.jpg
label 2
image_path /content/drive/MyDrive/food101_dataset/cheesecake/3900431.jpg
label 2
image_path /content/drive/MyDrive/food101_dataset/cheesecake/3406925.jpg
label 2
image_path /content/drive/MyDrive/food101_dataset/hot_dog/907088.jpg
label 1
image_path /content/drive/MyDrive/food101_dataset/chocolate_cake/1104843.jpg
label 2
image_path /content/drive/MyDrive/food101_dataset/cheesecake/3189636.jpg
label 2
image_path /content/drive/MyDrive/food101_dataset/chocolate_cake/3021534.jpg
label 2
image_path /content/drive/MyDrive/food101_dataset/cheesecake/361721.jpg
label 2
image_path /content/drive/MyDrive/food101_dataset/chocolate_cake/800835.

/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Streaming output truncated to the last 5000 lines.
image_path /content/drive/MyDrive/food101_dataset/hot_dog/2394823.jpg
label 1
image_path /content/drive/MyDrive/food101_dataset/hamburger/1181766.jpg
label 0
image_path /content/drive/MyDrive/food101_dataset/chocolate_cake/1656529.jpg
label 2
image_path /content/drive/MyDrive/food101_dataset/hot_dog/2140206.jpg
label 1
image_path /content/drive/MyDrive/food101_dataset/hot_dog/2063635.jpg
label 1
image_path /content/drive/MyDrive/food101_dataset/hot_dog/1831563.jpg
label 1
image_path /content/drive/MyDrive/food101_dataset/hot_dog/2977401.jpg
label 1
image_path /content/drive/MyDrive/food101_dataset/hamburger/2241307.jpg
label 0
image_path /content/drive/MyDrive/food101_dataset/hamburger/244216.jpg
label 0
image_path /content/drive/MyDrive/food101_dataset/cheesecake/3490386.jpg
label 2
image_path /content/drive/MyDrive/food101_dataset/chocolate_cake/2433547.jpg
label2 
image_path /content/drive/MyDrive/food101_dataset/hot_dog/1143110.jpg

/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Streaming output truncated to the last 5000 lines.
labellabel  10

image_pathimage_path /content/drive/MyDrive/food101_dataset/cheesecake/3020434.jpg 
/content/drive/MyDrive/food101_dataset/hot_dog/987193.jpglabel 
2
label 1
image_path /content/drive/MyDrive/food101_dataset/hot_dog/3842701.jpgimage_path
 label/content/drive/MyDrive/food101_dataset/hot_dog/3105556.jpg
 label1 
1
image_path /content/drive/MyDrive/food101_dataset/chocolate_cake/3516897.jpg
label 2
image_path /content/drive/MyDrive/food101_dataset/cheesecake/1693775.jpg
label 2
image_path /content/drive/MyDrive/food101_dataset/chocolate_cake/3783778.jpgimage_path

label  2/content/drive/MyDrive/food101_dataset/cheesecake/3031568.jpg
label 2
image_path /content/drive/MyDrive/food101_dataset/chocolate_cake/2280445.jpg
label 2
image_path /content/drive/MyDrive/food101_dataset/cheesecake/212691.jpglabel
image_path  /content/drive/MyDrive/food101_dataset/hamburger/245244.jpg
label2 0

image_path /content/drive/MyDrive/food101_d

...

Streaming output truncated to the last 5000 lines.
labelimage_path  /content/drive/MyDrive/food101_dataset/cheesecake/385675.jpg1

label 2image_path 
/content/drive/MyDrive/food101_dataset/kebab/Image_38.jpg
label 3
image_path /content/drive/MyDrive/food101_dataset/hamburger/903277.jpg
label 0
image_path /content/drive/MyDrive/food101_dataset/hamburger/1237509.jpg
label 0
image_path /content/drive/MyDrive/food101_dataset/chocolate_cake/2584547.jpg
label 2
image_path /content/drive/MyDrive/food101_dataset/cheesecake/570321.jpg
label 2
image_path /content/drive/MyDrive/food101_dataset/hamburger/231857.jpg
label 0
image_path /content/drive/MyDrive/food101_dataset/chocolate_cake/2383373.jpgimage_path
label /content/drive/MyDrive/food101_dataset/cheesecake/369347.jpg 2
label
 2
image_path image_path/content/drive/MyDrive/food101_dataset/chocolate_cake/1086272.jpg
label /content/drive/MyDrive/food101_dataset/chocolate_cake/2747767.jpg 
2
label 2
image_path /content/drive/MyDrive/food101_data

/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


image_path /content/drive/MyDrive/food101_dataset/chocolate_cake/1012759.jpg
label 2
image_path /content/drive/MyDrive/food101_dataset/chocolate_cake/3005657.jpg
image_pathlabel  /content/drive/MyDrive/food101_dataset/cheesecake/3834149.jpg
2label 2

image_path image_path/content/drive/MyDrive/food101_dataset/chocolate_cake/2533462.jpg
 /content/drive/MyDrive/food101_dataset/chocolate_cake/3121457.jpglabel 
2label
 2
image_path image_path/content/drive/MyDrive/food101_dataset/hot_dog/2062787.jpg 
/content/drive/MyDrive/food101_dataset/chocolate_cake/2835003.jpglabel
 label1 
2
image_pathimage_path  /content/drive/MyDrive/food101_dataset/chocolate_cake/53320.jpg/content/drive/MyDrive/food101_dataset/chocolate_cake/251994.jpg

labellabel  22

image_path /content/drive/MyDrive/food101_dataset/chocolate_cake/2304038.jpg
label 2
image_pathimage_path  /content/drive/MyDrive/food101_dataset/cheesecake/2637470.jpg/content/drive/MyDrive/food101_dataset/hot_dog/14043.jpg

labellabel  21

image_p

/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Streaming output truncated to the last 5000 lines.
image_path /content/drive/MyDrive/food101_dataset/chocolate_cake/1721926.jpgimage_path
 label /content/drive/MyDrive/food101_dataset/hamburger/3807701.jpg2
label
 0
image_path image_path/content/drive/MyDrive/food101_dataset/cheesecake/946930.jpg 
/content/drive/MyDrive/food101_dataset/cheesecake/1159749.jpglabel
 label2 
2
image_path image_path /content/drive/MyDrive/food101_dataset/chocolate_cake/295007.jpg/content/drive/MyDrive/food101_dataset/hamburger/1964319.jpg

labellabel 0 
2
image_pathimage_path  /content/drive/MyDrive/food101_dataset/chocolate_cake/2216578.jpg
/content/drive/MyDrive/food101_dataset/chocolate_cake/2398993.jpglabel 
2label
 2
image_pathimage_path  /content/drive/MyDrive/food101_dataset/hamburger/3912017.jpg/content/drive/MyDrive/food101_dataset/chocolate_cake/1646662.jpg

label label0 
2
image_path /content/drive/MyDrive/food101_dataset/chocolate_cake/3707840.jpgimage_path 
label/content/drive/MyDrive/food101_

# **menu_detector model usage**